In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
#####Cross-Attention
# -- Squeeze-and-Excitation (SE) Block --
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        w = self.pool(x).view(b, c)
        w = self.fc(w).view(b, c, 1)
        return x * w

# -- Gated Convolution Block --
class GatedConv1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2)
        self.gate = nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2)
        self.bn = nn.BatchNorm1d(out_channels)

    def forward(self, x):
        feature = self.conv(x)
        gate = torch.sigmoid(self.gate(x))
        return self.bn(feature * gate)

# -- Quadratic Convolution Block --
class QuadraticConv1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size):
        super().__init__()
        self.linear = nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2)
        self.quadratic = nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2)

    def forward(self, x):
        return self.linear(x) + self.quadratic(x) ** 2

# -- Residual Block with SE and GatedConv --
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=7):
        super().__init__()
        self.gconv1 = GatedConv1D(in_channels, out_channels, kernel_size)
        self.gconv2 = GatedConv1D(out_channels, out_channels, kernel_size)
        self.se = SEBlock(out_channels)
        self.res = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        res = self.res(x)
        x = self.gconv1(x)
        x = self.gconv2(x)
        x = self.se(x)
        return F.relu(x + res)

# -- Nonlinear Fusion --
class NonlinearFusion(nn.Module):
    def __init__(self, in_dim=512, out_dim=128):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, in_dim)
        self.gate = nn.Linear(in_dim, in_dim)
        self.act = nn.SiLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(in_dim, out_dim)

    def forward(self, x):
        gated = torch.sigmoid(self.gate(x))
        x = self.act(self.fc1(x)) * gated
        x = self.dropout(x)
        return self.fc2(x)


# Cross-Attention Layer
class CrossAttention(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=heads, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Linear(dim, dim)
        )

    def forward(self, query, key, value):
        # Compute the attention output
        attn_out, _ = self.attn(query, key, value)  # (B, T, dim)
        
        # Process the attention output further if needed
        attn_out = self.fc(attn_out)
        
        return attn_out

# GlobalEncoder with Cross-Attention
class GlobalEncoderWithAttention(nn.Module):
    def __init__(self, in_channels=12, hidden_dim=128, heads=4):
        super().__init__()
        self.res1 = ResidualBlock(in_channels, 64)
        self.res2 = ResidualBlock(64, 128)
        self.lstm = nn.LSTM(128, hidden_dim, batch_first=True, bidirectional=True)
        self.attn = CrossAttention(dim=2*hidden_dim, heads=heads)

    def forward(self, x):  # x: (B, T, C)
        x = x.permute(0, 2, 1)  # (B, C, T)
        x = self.res1(x)
        x = self.res2(x)        # (B, 128, T)
        x = x.permute(0, 2, 1)  # (B, T, 128)
        x, _ = self.lstm(x)  # LSTM output: (B, T, 2*hidden_dim)
        return x

# LocalEncoder with Cross-Attention
class LocalEncoderWithAttention(nn.Module):
    def __init__(self, in_channels=12, hidden_dim=128, heads=4):
        super().__init__()
        self.res1 = ResidualBlock(in_channels, 64)
        self.res2 = ResidualBlock(64, 128)
        self.lstm = nn.LSTM(128, hidden_dim, batch_first=True, bidirectional=True)
        self.attn = CrossAttention(dim=2*hidden_dim, heads=heads)

    def forward(self, x):  # x: (B, S, T', C)
        B, S, T, C = x.shape
        x = x.view(B*S, T, C).permute(0, 2, 1)  # (B*S, C, T)
        x = self.res1(x)
        x = self.res2(x)                        # (B*S, 128, T)
        x = x.permute(0, 2, 1)                  # (B*S, T, 128)
        x, _ = self.lstm(x)                    # (B*S, T, 2*hidden_dim)
        return x

# Full Model with Cross-Attention Fusion
class CardioformerECGWithCrossAttention(nn.Module):
    def __init__(self, num_supercls=5, num_subcls=20, heads=4):
        super().__init__()
        self.global_encoder = GlobalEncoderWithAttention(heads=heads)
        self.local_encoder = LocalEncoderWithAttention(heads=heads)
        self.fusion = NonlinearFusion(in_dim=512, out_dim=128)
        self.head_super = nn.Linear(128, num_supercls)
        self.head_sub = nn.Linear(128, num_subcls)

    def forward(self, global_input, feature_input):
        B, T, C = global_input.shape
        Bf, Nf, Tf, Cf = feature_input.shape

        # Extract features from the global and local encoders
        g_feat = self.global_encoder(global_input)  # (B, T, 2*hidden_dim)
        l_feat = self.local_encoder(feature_input)    # (B*S, T, 2*hidden_dim)

        # Apply cross-attention between global and local features
        # Global features are used as the query, and local features as the key and value
        cross_attn_out = self.attn(g_feat, l_feat, l_feat)  # (B, T, 2*hidden_dim)

        # Global + Local Fusion (concatenating after attention)
        fused = torch.mean(cross_attn_out, dim=1)  # (B, 2*hidden_dim)
        
        # Further processing with the nonlinear fusion layer
        fused = self.fusion(fused)
        
        # Predict super-class and sub-class labels
        return self.head_super(fused), self.head_sub(fused)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
####Temperal + Multi-Scale Convolution Blocl 
# -- Temporal Attention Block --
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        w = self.pool(x).view(b, c)
        w = self.fc(w).view(b, c, 1)
        return x * w

# -- Gated Convolution Block --
class GatedConv1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2)
        self.gate = nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2)
        self.bn = nn.BatchNorm1d(out_channels)

    def forward(self, x):
        feature = self.conv(x)
        gate = torch.sigmoid(self.gate(x))
        return self.bn(feature * gate)

# -- Quadratic Convolution Block --
class QuadraticConv1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size):
        super().__init__()
        self.linear = nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2)
        self.quadratic = nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2)

    def forward(self, x):
        return self.linear(x) + self.quadratic(x) ** 2

# -- Residual Block with SE and GatedConv --
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=7):
        super().__init__()
        self.gconv1 = GatedConv1D(in_channels, out_channels, kernel_size)
        self.gconv2 = GatedConv1D(out_channels, out_channels, kernel_size)
        self.se = SEBlock(out_channels)
        self.res = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        res = self.res(x)
        x = self.gconv1(x)
        x = self.gconv2(x)
        x = self.se(x)
        return F.relu(x + res)

# -- Nonlinear Fusion --
class NonlinearFusion(nn.Module):
    def __init__(self, in_dim=512, out_dim=128):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, in_dim)
        self.gate = nn.Linear(in_dim, in_dim)
        self.act = nn.SiLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(in_dim, out_dim)

    def forward(self, x):
        gated = torch.sigmoid(self.gate(x))
        x = self.act(self.fc1(x)) * gated
        x = self.dropout(x)
        return self.fc2(x)
    
class TemporalAttention(nn.Module):
    def __init__(self, in_channels, heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=in_channels, num_heads=heads, batch_first=True)

    def forward(self, x):
        return self.attn(x, x, x)[0]

# -- Add Multi-Scale Convolution Block --
class MultiScaleConv1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_sizes=[3, 5, 7]):
        super().__init__()
        self.convs = nn.ModuleList([nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2) for kernel_size in kernel_sizes])
        self.bn = nn.BatchNorm1d(out_channels * len(kernel_sizes))

    def forward(self, x):
        x = torch.cat([conv(x) for conv in self.convs], dim=1)  # Concatenate along channel dimension
        return self.bn(x)

# -- Gated Multi-Head Attention 1D --
class GatedMultiHeadAttention1D(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=heads, batch_first=True)
        self.gate = nn.Sequential(
            nn.Linear(dim, dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        attn_out = self.attn(x, x, x)[0]
        gate = self.gate(x)
        return attn_out * gate
    
# -- Global Encoder with Temporal Attention --
class GlobalEncoderWithAttention(nn.Module):
    def __init__(self, in_channels=12, hidden_dim=128):
        super().__init__()
        self.res1 = ResidualBlock(in_channels, 64)
        self.res2 = ResidualBlock(64, 128)
        self.lstm = nn.LSTM(128, hidden_dim, batch_first=True, bidirectional=True)
        self.attn = GatedMultiHeadAttention1D(dim=2*hidden_dim, heads=4)
        self.temporal_attn = TemporalAttention(hidden_dim * 2)  # Bidirectional LSTM

    def forward(self, x):  # x: (B, T, C)
        x = x.permute(0, 2, 1)  # (B, C, T)
        x = self.res1(x)
        x = self.res2(x)        # (B, 128, T)
        x = x.permute(0, 2, 1)  # (B, T, 128)
        x, _ = self.lstm(x)
        x = self.temporal_attn(x)  # Apply temporal attention
        x = self.attn(x)
        x = torch.mean(x, dim=1)  # (B, 2*hidden_dim)
        return x

# -- Local Encoder with Multi-Scale Convolutions --
class LocalEncoderWithMultiScaleConv(nn.Module):
    def __init__(self, in_channels=12, hidden_dim=128):
        super().__init__()
        self.res1 = ResidualBlock(in_channels, 64)
        self.res2 = ResidualBlock(64, 128)
        self.multi_scale_conv = MultiScaleConv1D(128, 128)  # Multi-Scale Convolution Block
        self.lstm = nn.LSTM(128, hidden_dim, batch_first=True, bidirectional=True)
        self.attn = GatedMultiHeadAttention1D(dim=2*hidden_dim, heads=4)

    def forward(self, x):  # x: (B, S, T', C)
        B, S, T, C = x.shape
        x = x.view(B*S, T, C).permute(0, 2, 1)  # (B*S, C, T)
        x = self.res1(x)
        x = self.res2(x)                        # (B*S, 128, T)
        x = self.multi_scale_conv(x)            # Apply Multi-Scale Convolution
        x = x.permute(0, 2, 1)                  # (B*S, T, 128)
        x, _ = self.lstm(x)
        x = self.attn(x)
        x = torch.mean(x, dim=1)                # (B*S, 2*hidden_dim)
        x = x.view(B, S, -1)
        x = torch.mean(x, dim=1)                # (B, 2*hidden_dim)
        return x

# -- Full Cardioformer-Style Model --
class CardioformerECGWithEnhancements(nn.Module):
    def __init__(self, num_supercls=5, num_subcls=20):
        super().__init__()
        self.global_encoder = GlobalEncoderWithAttention()
        self.local_encoder = LocalEncoderWithMultiScaleConv()
        self.fusion = NonlinearFusion(in_dim=512, out_dim=128)
        self.head_super = nn.Linear(128, num_supercls)
        self.head_sub = nn.Linear(128, num_subcls)

    def forward(self, global_input, feature_input):
        B, T, C = global_input.shape
        Bf, Nf, Tf, Cf = feature_input.shape
        g_feat = self.global_encoder(global_input)  # (B, 2*H)
        l_feat = self.local_encoder(feature_input)    # (B, 2*H)
        fused = torch.cat([g_feat, l_feat], dim=1)  # (B, 4*H = 512)
        fused = self.fusion(fused)
        return self.head_super(fused), self.head_sub(fused)
